# HTC (Hybrid Task Cascade) Training Results - Batch 2 - Automatic Extraction
**All data retrieved from htc_training_batch2 folder**

In [3]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Load All Data from Training Folder

In [4]:
# Paths
WORK_DIR = Path('htc_batch2')
training_dirs = [d for d in WORK_DIR.glob('202*') if d.is_dir()]
latest_dir = sorted(training_dirs)[-1]
scalars_file = latest_dir / 'vis_data' / 'scalars.json'
log_file = list(latest_dir.glob('*.log'))[0]

print(f"📂 Training directory: {latest_dir}")
print(f"📄 Scalars file: {scalars_file.exists()}")
print(f"📄 Log file: {log_file.exists()}")

📂 Training directory: htc_batch2\20260509_221050
📄 Scalars file: True
📄 Log file: True


In [5]:
# Load training data from scalars.json
with open(scalars_file, 'r') as f:
    all_data = [json.loads(line) for line in f]

train_data = [d for d in all_data if 'loss' in d]
train_df = pd.DataFrame(train_data)

print(f"✅ Loaded {len(train_df)} training iterations")
print(f"📊 Epochs: {train_df['epoch'].min()} to {train_df['epoch'].max()}")
print(f"📊 Total iterations: {train_df['step'].max()}")

✅ Loaded 2088 training iterations
📊 Epochs: 1 to 36
📊 Total iterations: 105660


In [6]:
# Extract final validation results from log
with open(log_file, 'r') as f:
    log_content = f.read()
    log_lines = log_content.split('\n')

# Find bbox results
bbox_line = [l for l in log_lines if 'bbox_mAP_copypaste' in l][-1]
bbox_values = bbox_line.split('bbox_mAP_copypaste:')[1].strip().split()
bbox_mAP, bbox_mAP_50, bbox_mAP_75, bbox_mAP_s, bbox_mAP_m, bbox_mAP_l = map(float, bbox_values)

# Find segm results
segm_line = [l for l in log_lines if 'segm_mAP_copypaste' in l][-1]
segm_values = segm_line.split('segm_mAP_copypaste:')[1].strip().split()
segm_mAP, segm_mAP_50, segm_mAP_75, segm_mAP_s, segm_mAP_m, segm_mAP_l = map(float, segm_values)

print("✅ Extracted final results:")
print(f"   bbox mAP: {bbox_mAP:.3f}")
print(f"   segm mAP: {segm_mAP:.3f}")

✅ Extracted final results:
   bbox mAP: 0.728
   segm mAP: 0.702


## 2. Final Performance Results

In [7]:
# BBox results
bbox_df = pd.DataFrame({
    'Metric': ['mAP', 'mAP@50', 'mAP@75', 'mAP (small)', 'mAP (medium)', 'mAP (large)'],
    'Value': [bbox_mAP, bbox_mAP_50, bbox_mAP_75, bbox_mAP_s, bbox_mAP_m, bbox_mAP_l],
    'Percentage': [f"{v*100:.1f}%" for v in [bbox_mAP, bbox_mAP_50, bbox_mAP_75, bbox_mAP_s, bbox_mAP_m, bbox_mAP_l]]
})

# Segm results
segm_df = pd.DataFrame({
    'Metric': ['mAP', 'mAP@50', 'mAP@75', 'mAP (small)', 'mAP (medium)', 'mAP (large)'],
    'Value': [segm_mAP, segm_mAP_50, segm_mAP_75, segm_mAP_s, segm_mAP_m, segm_mAP_l],
    'Percentage': [f"{v*100:.1f}%" for v in [segm_mAP, segm_mAP_50, segm_mAP_75, segm_mAP_s, segm_mAP_m, segm_mAP_l]]
})

print("="*60)
print("BBOX DETECTION RESULTS")
print("="*60)
print(bbox_df[['Metric', 'Percentage']].to_string(index=False))

print("\n" + "="*60)
print("INSTANCE SEGMENTATION RESULTS")
print("="*60)
print(segm_df[['Metric', 'Percentage']].to_string(index=False))

BBOX DETECTION RESULTS
      Metric Percentage
         mAP      72.8%
      mAP@50      92.4%
      mAP@75      79.6%
 mAP (small)      28.7%
mAP (medium)      62.5%
 mAP (large)      81.1%

INSTANCE SEGMENTATION RESULTS
      Metric Percentage
         mAP      70.2%
      mAP@50      90.5%
      mAP@75      76.5%
 mAP (small)      19.1%
mAP (medium)      57.2%
 mAP (large)      80.3%


## 3. Export Results

In [8]:
# Create results directory
results_dir = Path('results_batch2')
results_dir.mkdir(exist_ok=True)

# Export summary table
summary = pd.DataFrame({
    'Method': ['HTC Batch 2'],
    'bbox_mAP': [f"{bbox_mAP*100:.1f}"],
    'bbox_mAP_50': [f"{bbox_mAP_50*100:.1f}"],
    'bbox_mAP_75': [f"{bbox_mAP_75*100:.1f}"],
    'segm_mAP': [f"{segm_mAP*100:.1f}"],
    'segm_mAP_50': [f"{segm_mAP_50*100:.1f}"],
    'segm_mAP_75': [f"{segm_mAP_75*100:.1f}"]
})

summary.to_csv(results_dir / 'htc_batch2_summary.csv', index=False)
bbox_df.to_csv(results_dir / 'htc_batch2_bbox.csv', index=False)
segm_df.to_csv(results_dir / 'htc_batch2_segm.csv', index=False)

print("✅ Results exported to:")
print(f"   - {results_dir / 'htc_batch2_summary.csv'}")
print(f"   - {results_dir / 'htc_batch2_bbox.csv'}")
print(f"   - {results_dir / 'htc_batch2_segm.csv'}")

✅ Results exported to:
   - results_batch2\htc_batch2_summary.csv
   - results_batch2\htc_batch2_bbox.csv
   - results_batch2\htc_batch2_segm.csv


---
**All data automatically extracted from:** `htc_training_batch2/htc/`